# Cycle 6 - Allègement du modèle pour la mise en production

Le modèle du cycle 5 (200 arbres) occupe environ **950 Mo de RAM** une fois chargé par l'API. L'hébergeur visé (Render, offre gratuite) limite le service à **512 Mo**.

Objectif : trouver une Random Forest plus légère qui garde presque les mêmes performances, puis la sauvegarder pour l'API.

In [3]:
import json
import pickle
from datetime import datetime as dt
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

import notebooks.utils as utils
# Import par le chemin complet : le modèle sauvegardé référence cette classe, l'API doit pouvoir la ré-importer
from notebooks.agritech_feature_engineering import AgritechFeatureEngineering

In [4]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Agriculture_optimization")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1795094493928, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795094493928, lifecycle_stage='active', name='Agriculture_optimization', tags={}, trace_location=None, workspace='default'>

## Données

Mêmes colonnes et même découpage temporel que dans les cycles précédents.

In [5]:
dataset = pd.read_csv('../data/rafined/dataset.csv')

INPUT_FEATURES = [
    'Area', 'Item',
    'avg_temp', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'pesticides_kg_ha',
    'missing_pesticide_value', 'missing_rain_value', 'missing_temp_value',
]
CULTURES = sorted(dataset['Item'].unique())

X_train, X_test, y_train, y_test = utils.test_train_time_split(dataset)
X_train = X_train.sort_values('Year')  # indispensable pour TimeSeriesSplit
y_train = y_train.loc[X_train.index]
X_train, X_test = X_train[INPUT_FEATURES], X_test[INPUT_FEATURES]

Année de césure calculée : 2009
Proportion réelle train : 78.73% (20349 lignes)
Proportion réelle test  : 21.27% (5499 lignes)


## Pipeline

Même pipeline qu'au cycle 5. On garde les hyperparamètres trouvés au cycle 4, sauf le nombre d'arbres et la taille minimale des feuilles, qui sont les deux leviers pour alléger le modèle.

Pour estimer la place d'un modèle en RAM, on le sérialise sans compression et on mesure sa taille.

In [6]:
# Hyperparamètres du cycle 4 qu'on ne change pas
BASE_PARAMS = {'max_features': 0.5, 'max_depth': 40, 'bootstrap': False}


def build_model(params):
    """Pipeline complet : feature engineering, prétraitement, Random Forest sur cible log1p."""
    numeric_features = INPUT_FEATURES[2:] + ['aridity', 'avg_temp_square']

    preprocessor = ColumnTransformer([
        ('area_encoding', TargetEncoder(target_type='continuous', cv=KFold(5, shuffle=True, random_state=42)), ['Area']),
        ('item_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Item']),
        ('num_pipeline', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
    ])

    pipeline = Pipeline([
        ('features', AgritechFeatureEngineering()),
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(**BASE_PARAMS, **params, random_state=42, n_jobs=-1)),
    ])

    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)


def model_size_mb(model):
    """Taille du modèle sérialisé sans compression (≈ place en RAM), en Mo."""
    return len(pickle.dumps(model)) / 1e6

## Comparaison de variantes plus légères

Pour rappel, le modèle du cycle 5 (200 arbres, feuilles d'une seule ligne) pèse **566 Mo** pour une RMSE de **24 978 hg/ha** en validation croisée (cycle 4).

On teste trois variantes, évaluées comme au cycle 4 en validation croisée temporelle sur le train uniquement :
- **100 arbres** au lieu de 200 : la taille est divisée par deux ;
- **feuilles ≥ 2 ou ≥ 5** : chaque feuille doit regrouper plusieurs lignes, les arbres ont donc beaucoup moins de nœuds.

⏱️ Cette cellule prend quelques minutes.

In [7]:
candidats = {
    '100 arbres': {'n_estimators': 100, 'min_samples_leaf': 1},
    '100 arbres, feuilles ≥ 2': {'n_estimators': 100, 'min_samples_leaf': 2},
    '100 arbres, feuilles ≥ 5': {'n_estimators': 100, 'min_samples_leaf': 5},
}

resultats = []
for nom, params in candidats.items():
    model = build_model(params)
    scores = cross_validate(model, X_train, y_train, cv=TimeSeriesSplit(5), scoring='neg_root_mean_squared_error')
    model.fit(X_train, y_train)
    resultats.append({
        'variante': nom,
        'taille_mo': model_size_mb(model),
        'rmse_cv': -scores['test_score'].mean(),
    })

resultats = pd.DataFrame(resultats)
resultats.round(0)

,variante,taille_mo,rmse_cv
0,100 arbres,283.0,24976.0
1,"100 arbres, feuilles ≥ 2",124.0,25087.0
2,"100 arbres, feuilles ≥ 5",46.0,26148.0


## Choix du modèle

On se fixe un **budget de 250 Mo** pour le modèle : avec Python, pandas, scikit-learn et FastAPI (environ 150 Mo), l'API reste nettement sous les 512 Mo de Render.

- 100 arbres seuls ne suffisent pas : le modèle dépasse encore le budget.
- Avec des **feuilles d'au moins 2 lignes**, le modèle passe sous le budget et la RMSE ne bouge presque pas : c'est la variante retenue.
- Avec des feuilles d'au moins 5 lignes, le modèle est encore plus léger, mais la RMSE se dégrade nettement : ce n'est pas nécessaire.

In [8]:
CHOSEN_PARAMS = candidats['100 arbres, feuilles ≥ 2']

## Évaluation sur le jeu de test (2009 à 2013)

Le modèle allégé est entraîné sur le train puis évalué une seule fois sur le test. On le compare aux scores obtenus sur ce même test par la baseline naïve et par le modèle du cycle 5.

In [9]:
light_model = build_model(CHOSEN_PARAMS).fit(X_train, y_train)
test_metrics = utils.evaluate(y_test, light_model.predict(X_test))

pd.DataFrame({
    'Baseline naïve (cycle 0)': {'r2': 0.836, 'mae': 16461, 'rmse': 32646},
    'Cycle 5 (200 arbres)': {'r2': 0.852, 'mae': 15808, 'rmse': 31012},
    'Cycle 6 (allégé)': test_metrics,
}).T.round(3)

,r2,mae,rmse
Baseline naïve (cycle 0),0.836,16461.000,32646.00
Cycle 5 (200 arbres),0.852,15808.000,31012.00
Cycle 6 (allégé),0.854,15886.098,30871.32


## Les recommandations restent-elles cohérentes ?

On regarde les 3 cultures recommandées pour trois pays avec leur contexte réel de 2013. D'après l'analyse des données, on s'attend à retrouver en tête les cultures au fort rendement par hectare : tubercules (pomme de terre, manioc, igname, patate douce) et plantains.

In [10]:
def top3(model, pays, annee=2013):
    """Les 3 cultures avec le meilleur rendement prédit pour le contexte réel d'un pays."""
    contexte = dataset[(dataset['Area'] == pays) & (dataset['Year'] == annee)].iloc[0][INPUT_FEATURES].to_dict()
    lignes = pd.DataFrame([{**contexte, 'Item': culture} for culture in CULTURES])[INPUT_FEATURES]
    predictions = pd.Series(model.predict(lignes), index=CULTURES)
    return ', '.join(predictions.nlargest(3).index)


for pays in ['France', 'India', 'Kenya']:
    print(f"{pays} : {top3(light_model, pays)}")

France : Pomme de terre, Patate douce, Igname
India : Manioc, Pomme de terre, Plantains et autres
Kenya : Pomme de terre, Plantains et autres, Patate douce


## Modèle de production

L'évaluation est terminée : le modèle retenu est réentraîné sur toutes les années (1990 à 2013) pour profiter des données les plus récentes. Les performances annoncées restent celles mesurées sur le test.

In [11]:
X_all, y_all = dataset[INPUT_FEATURES], dataset['hg/ha']
final_model = build_model(CHOSEN_PARAMS).fit(X_all, y_all)

final_size_mb = model_size_mb(final_model)
print(f"Taille en mémoire du modèle de production : {final_size_mb:.0f} Mo (budget : 250 Mo)")

Taille en mémoire du modèle de production : 158 Mo (budget : 250 Mo)


## Sauvegarde pour l'API

Deux fichiers sont enregistrés dans `models/` : le pipeline complet (`agriculture_model.joblib`) et ses métadonnées (`agriculture_model_metadata.json`), qui ne contiennent que ce que l'API et le dashboard utilisent.

In [12]:
model_path = Path('../models/agriculture_model.joblib')
metadata_path = Path('../models/agriculture_model_metadata.json')

metadata = {
    'version': dt.now().strftime("%Y%m%d_%H%M%S"),
    'cultures': CULTURES,
    'pays': sorted(dataset['Area'].unique()),
    # cultures observées dans chaque pays : la recommandation ne propose que celles-ci
    'cultures_par_pays': {pays: sorted(items.unique()) for pays, items in dataset.groupby('Area')['Item']},
    # bornes des variables numériques, utilisées pour les sliders du dashboard
    'feature_ranges': {
        col: {
            'min': round(float(dataset[col].min()), 2),
            'median': round(float(dataset[col].median()), 2),
            'max': round(float(dataset[col].max()), 2),
        }
        for col in ['avg_temp', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'pesticides_kg_ha']
    },
    'training_years': [int(dataset['Year'].min()), int(dataset['Year'].max())],
    'evaluation': {'modele': {k: round(float(v), 4) for k, v in test_metrics.items()}},
}

joblib.dump(final_model, model_path, compress=3)
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Vérification : le modèle rechargé donne les mêmes prédictions
reloaded = joblib.load(model_path)
assert np.allclose(reloaded.predict(X_test.head()), final_model.predict(X_test.head()))

print(f"Modèle enregistré : {model_path} ({model_path.stat().st_size / 1e6:.0f} Mo sur disque)")

Modèle enregistré : ../models/agriculture_model.joblib (48 Mo sur disque)


In [13]:
with mlflow.start_run(run_name="Cycle 6 - Allègement du modèle"):
    mlflow.set_tag('cycle', '6_allegement')
    mlflow.log_params({**BASE_PARAMS, **CHOSEN_PARAMS})
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})
    mlflow.log_metric('taille_mo', final_size_mb)
    mlflow.log_table(resultats, artifact_file='comparaison_variantes.json')

🏃 View run Cycle 6 - Allègement du modèle at: http://127.0.0.1:5000/#/experiments/1/runs/5c9a86298e1743378a30e8494710747c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Conclusion du cycle 6

- En passant de 200 à 100 arbres et en imposant au moins 2 lignes par feuille, le modèle passe d'environ **566 Mo à 124 Mo** en mémoire (−78 %), pour une RMSE de validation quasiment identique (+0,4 %).
- Réentraîné sur toutes les années, le modèle de production pèse environ **158 Mo** : il tient dans le budget de 250 Mo, donc dans les 512 Mo de Render.
- Les scores sur le test et les recommandations ci-dessus permettent de vérifier que l'allègement ne change pas la qualité du modèle pour l'utilisateur.

C'est ce modèle allégé qui est utilisé par l'API.